### Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code

In [13]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown,display

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')


In [36]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)


In [37]:
OPENAI_MODEL = "gpt-5-nano"
GEMINI_MODEL = "gemini-2.5-flash-lite"

In [17]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '24.6.0',
  'version': 'Darwin Kernel Version 24.6.0: Mon Jan 19 22:01:41 PST 2026; root:xnu-11417.140.69.708.3~1/RELEASE_ARM64_T8132',
  'kernel': '24.6.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin24.6.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M4',
  'cores_logical': 10,
  'cores_physical': 10,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 17.0.0 (clang-1700.6.3.2)',
   'g++': 'Apple clang version 17.0.0 (clang-1700.6.3.2)',
   'clang': 'Apple clang version 17.0.0 (clang-1700.6.3.2)',
   'msvc_cl': ''},
  'build_tools': {'cmake': 'cmake version 4.2.0',
   'ninja': '',
   'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [18]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))
    

Short answer: You don’t need to install anything new. Your system already has a C++ compiler (Apple clang via Xcode Command Line Tools) and basic build tools.

What you have (from your report)
- C++ compiler: Apple clang version 17.0.0 (clang-1700.6.3.2)
- Build tools: make, cmake
- Command Line Tools (CLT) for Xcode is available

If you’re set up, here are the simplest ways to build and run main.cpp

A) Manually in the Terminal (fastest, no extra steps)
- Open Terminal
- Navigate to the folder with main.cpp:
  - cd /path/to/your/file
- Compile (fastest runtime with reasonable safety and modern C++ features):
  - clang++ -std=c++20 -O3 -DNDEBUG main.cpp -o main
- Run:
  - ./main

Notes:
- -std=c++20 selects a modern C++ standard; use -std=c++17 or -std=c++23 if your code requires it.
- -O3 enables aggressive optimization for speed.
- -DNDEBUG disables runtime debug checks (optional for speed).

B) If you want to do it from Python (using your template)

compile_command = ["clang++", "-std=c++20", "-O3", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

Then in Python:
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout

Tips and alternatives
- If your code doesn’t need C++20 features, you can use -std=c++17 or -std=c++23 depending on what your code uses.
- If you prefer g++/gcc syntax, clang++ and g++ are interchangeable on macOS for this purpose; you can use g++ instead of clang++ in the commands above.
- If you ever encounter a missing tool message, you can install Xcode Command Line Tools with:
  - xcode-select --install
  - This is the simplest path to ensure the compiler and basic tools are present.

In summary: No installation is required. Use clang++ as shown above to compile and run main.cpp, or use the provided Python commands for automation.

In [19]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [20]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [21]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [22]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [23]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [24]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [26]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [27]:
run_python(pi)

Result: 3.141592656089
Execution Time: 10.007888 seconds


In [28]:
port(openai, OPENAI_MODEL, pi)

In [29]:
# Use the commands from GPT 5

def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

In [31]:
compile_and_run()

Result: 3.141592656089
Execution Time: 0.078257 seconds

Result: 3.141592656089
Execution Time: 0.054737 seconds

Result: 3.141592656089
Execution Time: 0.053926 seconds



In [39]:
print(f"""
In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {19.178207/0.104241:.0f}X speedup
3rd place: GPT-5: {19.178207/0.082168:.0f}X speedup
2nd place: Grok 4: {19.178207/0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {19.178207/0.013314:.0f}X speedup
""")


In Ed's experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: 184X speedup
3rd place: GPT-5: 233X speedup
2nd place: Grok 4: 1060X speedup
1st place: Gemini 2.5 Pro: 1440X speedup

